# Exploring the grade-classification results

`results/` contains the per-patient predicted probabilities of the LGG/HGG classifiers on the synthetic *hard* phantom benchmark
(features from predicted masks or from reference masks; logistic regression or gradient boosting; cross-validated on the development
set and evaluated once on the test set). These are synthetic data: the notebook shows how to inspect such outputs, not BraTS performance.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from bratsgan.stats import expected_calibration_error
res = ROOT / "results"
sorted(p.name for p in res.glob("grade_hard_a5.*.csv"))

In [ ]:
rows = []
for src in ("pred", "gt"):
    for clf in ("logreg", "gboost"):
        for split in ("cv", "test"):
            d = pd.read_csv(res / f"grade_hard_a5.{src}_{clf}.{split}.csv")
            auc = roc_auc_score(d["y"], d["prob"])
            rows.append(dict(masks=src, classifier=clf, split=split, n=len(d), auc=round(auc, 3),
                             ece=round(expected_calibration_error(d["y"].values, d["prob"].values), 3)))
pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for a, split in zip(ax, ("cv", "test")):
    for src in ("pred", "gt"):
        d = pd.read_csv(res / f"grade_hard_a5.{src}_logreg.{split}.csv")
        fpr, tpr, _ = roc_curve(d["y"], d["prob"])
        a.plot(fpr, tpr, label=f"{src} masks (AUC {roc_auc_score(d['y'], d['prob']):.2f})")
    a.plot([0, 1], [0, 1], "k:", lw=.8); a.set_title({"cv": "cross-validated", "test": "test"}[split]); a.set_xlabel("false positive rate"); a.legend()
ax[0].set_ylabel("true positive rate"); plt.show()